# Exploration de `aggregate_to_lower_frequency` et `interpolate_to_higher_frequency`

Ce notebook illustre le comportement des deux méthodes de conversion de fréquence du `FrequencyConverter` :

1. **`aggregate_to_lower_frequency`** — agrégation de données mensuelles vers le trimestriel, avec focus sur le cas d'un trimestre incomplet (2 mois sur 3)
2. **`interpolate_to_higher_frequency`** — interpolation de données trimestrielles vers le mensuel, avec focus sur les effets de `ffill` / `bfill` en présence de NaN en début et fin de série (délais de publication)

Chaque méthode est testée sur des **séries temporelles simples** et des **données de panel**.

---

## 0. Imports & utilitaires

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from tsforecast.utils.frequency.converter import FrequencyConverter

converter = FrequencyConverter()

# Palette de couleurs
COLORS = {
    'original': '#4e79a7',
    'aggregated': '#e15759',
    'interpolated': '#59a14f',
    'nan': '#bab0ac',
    'ffill': '#f28e2b',
    'bfill': '#b07aa1',
}


def print_section(title: str) -> None:
    """Print a formatted section title."""
    print(f"\n{'═' * 60}")
    print(f"  {title}")
    print(f"{'═' * 60}")

---
## 1. `aggregate_to_lower_frequency` — Séries temporelles

### 1.1 Cas nominal : agrégation mensuelle → trimestrielle complète

On crée une série mensuelle sur 12 mois (4 trimestres complets) et on agrège avec différentes méthodes (`mean`, `sum`, `last`).

In [ ]:
# Création d'une série mensuelle sur 12 mois
dates_m = pd.date_range('2024-01-01', periods=12, freq='MS')
values = [100, 110, 105, 120, 115, 125, 130, 140, 135, 150, 145, 155]
series_m = pd.Series(values, index=dates_m, name='indicateur')

print("Série mensuelle originale :")
print(series_m.to_frame().T.to_string())

# Agrégation avec différentes méthodes
print_section("Agrégation mensuelle → trimestrielle")

for method in ['mean', 'sum', 'last', 'first', 'min', 'max']:
    result = converter.aggregate_to_lower_frequency(series_m, 'QE', method=method)
    print(f"\n  Méthode '{method}' :")
    for date, val in result.items():
        print(f"    {date.strftime('%Y-%m-%d')} → {val:.2f}")

In [ ]:
# Visualisation comparative
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=False)

for ax, method in zip(axes, ['mean', 'sum', 'last']):
    result = converter.aggregate_to_lower_frequency(series_m, 'QE', method=method)
    
    ax.bar(series_m.index, series_m.values, width=20, alpha=0.4,
           color=COLORS['original'], label='Mensuel (original)')
    ax.bar(result.index, result.values, width=60, alpha=0.6,
           color=COLORS['aggregated'], label=f'Trimestriel ({method})')
    ax.set_title(f"Agrégation '{method}'", fontweight='bold')
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=45)

fig.suptitle('Agrégation mensuelle → trimestrielle (trimestres complets)', fontweight='bold')
plt.tight_layout()
plt.show()

### 1.2 Cas critique : trimestre incomplet (2 mois sur 3)

On crée une série mensuelle qui s'arrête en **février 2025** : le Q1 2025 ne contient que 2 mois (janvier et février) au lieu de 3.

**Question clé** : comment `resample` gère-t-il un trimestre incomplet ? La valeur `mean` sera-t-elle calculée sur 2 mois seulement ? La `sum` sera-t-elle sous-estimée ?

In [ ]:
# Série mensuelle avec trimestre incomplet en fin de période
dates_incomplete = pd.date_range('2024-01-01', periods=14, freq='MS')  # Jan 2024 → Fev 2025
values_incomplete = list(range(100, 114))  # 14 valeurs
series_incomplete = pd.Series(values_incomplete, index=dates_incomplete, name='indicateur')

print("Série mensuelle (14 mois, dernier trimestre incomplet) :")
print(series_incomplete)

print_section("Agrégation avec trimestre incomplet")

for method in ['mean', 'sum', 'count']:
    result = converter.aggregate_to_lower_frequency(series_incomplete, 'QE', method=method)
    print(f"\n  Méthode '{method}' :")
    for date, val in result.items():
        # Identification du trimestre incomplet
        is_incomplete = date == result.index[-1]
        marker = '  ← INCOMPLET (2/3 mois)' if is_incomplete else ''
        print(f"    {date.strftime('%Y-%m-%d')} → {val:.2f}{marker}")

In [ ]:
# Comparaison visuelle : trimestre complet vs incomplet
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, method in zip(axes, ['mean', 'sum']):
    result = converter.aggregate_to_lower_frequency(series_incomplete, 'QE', method=method)
    
    # Barres mensuelles
    colors_monthly = [COLORS['original']] * 12 + [COLORS['ffill']] * 2
    ax.bar(series_incomplete.index, series_incomplete.values, width=20, alpha=0.5,
           color=colors_monthly)
    
    # Barres trimestrielles
    colors_quarterly = [COLORS['aggregated']] * (len(result) - 1) + [COLORS['nan']]
    ax.bar(result.index, result.values, width=60, alpha=0.6,
           color=colors_quarterly, edgecolor='black', linewidth=0.5)
    
    ax.set_title(f"Méthode '{method}' — dernier trimestre incomplet", fontweight='bold')
    
    patches = [
        mpatches.Patch(color=COLORS['original'], alpha=0.5, label='Mois complets'),
        mpatches.Patch(color=COLORS['ffill'], alpha=0.5, label='Mois du trimestre incomplet'),
        mpatches.Patch(color=COLORS['aggregated'], alpha=0.6, label='Trimestre complet (agrégé)'),
        mpatches.Patch(color=COLORS['nan'], alpha=0.6, label='Trimestre incomplet (agrégé)'),
    ]
    ax.legend(handles=patches, fontsize=7, loc='upper left')
    ax.tick_params(axis='x', rotation=45)

fig.suptitle('Impact d\'un trimestre incomplet sur l\'agrégation', fontweight='bold')
plt.tight_layout()
plt.show()

### 1.3 Analyse détaillée : `mean` vs `sum` sur trimestre incomplet

- **`mean`** : calcule la moyenne sur les mois disponibles (2 mois) → résultat mathématiquement correct mais basé sur moins d'observations
- **`sum`** : additionne les mois disponibles (2 mois) → résultat **sous-estimé** par rapport à un trimestre complet
- **`count`** : compte le nombre d'observations → permet de détecter les trimestres incomplets

In [ ]:
# Tableau récapitulatif
agg_mean = converter.aggregate_to_lower_frequency(series_incomplete, 'QE', method='mean')
agg_sum = converter.aggregate_to_lower_frequency(series_incomplete, 'QE', method='sum')
agg_count = converter.aggregate_to_lower_frequency(series_incomplete, 'QE', method='count')
agg_last = converter.aggregate_to_lower_frequency(series_incomplete, 'QE', method='last')

df_recap = pd.DataFrame({
    'mean': agg_mean,
    'sum': agg_sum,
    'count': agg_count,
    'last': agg_last,
})
df_recap.index = df_recap.index.strftime('%Y-Q%q')
df_recap['complet'] = ['Oui', 'Oui', 'Oui', 'Oui', 'Non (2/3)']

print("Récapitulatif de l'agrégation mensuelle → trimestrielle :")
display(df_recap)

### 1.4 Comparaison des positions `QS` vs `QE`

La position (`S` = start, `E` = end) détermine l'alignement des dates dans l'index trimestriel. Comparons les résultats.

In [ ]:
print_section("Comparaison QS vs QE")

for pos in ['QS', 'QE']:
    result = converter.aggregate_to_lower_frequency(series_incomplete, pos, method='mean')
    print(f"\n  Position '{pos}' (mean) :")
    for date, val in result.items():
        print(f"    {date.strftime('%Y-%m-%d')} → {val:.2f}")

---
## 2. `aggregate_to_lower_frequency` — Données de panel

On crée un panel avec 2 entités, chacune ayant des données mensuelles. L'une a un trimestre incomplet, l'autre non.

In [ ]:
# Création du panel
rng = np.random.default_rng(seed=42)

# Entité A : 14 mois (trimestre incomplet)
dates_a = pd.date_range('2024-01-01', periods=14, freq='MS')
df_a = pd.DataFrame({
    'ventes': rng.normal(100, 10, 14).cumsum() / 14 + 100,
    'cout': rng.normal(50, 5, 14).cumsum() / 14 + 50,
}, index=dates_a)
df_a['entity'] = 'entité_A'

# Entité B : 12 mois (trimestres complets)
dates_b = pd.date_range('2024-01-01', periods=12, freq='MS')
df_b = pd.DataFrame({
    'ventes': rng.normal(200, 15, 12).cumsum() / 12 + 200,
    'cout': rng.normal(80, 8, 12).cumsum() / 12 + 80,
}, index=dates_b)
df_b['entity'] = 'entité_B'

# Assemblage du panel
df_panel = pd.concat([df_a, df_b])
df_panel = df_panel.reset_index().rename(columns={'index': 'date'})
df_panel = df_panel.set_index(['entity', 'date'])

print(f"Dimensions du panel : {df_panel.shape}")
print(f"Entités : {df_panel.index.get_level_values('entity').unique().tolist()}")
print(f"\nEntité A — dernières dates :")
print(df_panel.xs('entité_A', level='entity').tail())
print(f"\nEntité B — dernières dates :")
print(df_panel.xs('entité_B', level='entity').tail())

In [ ]:
# Agrégation du panel par entité
print_section("Agrégation du panel mensuel → trimestriel")

for entity in ['entité_A', 'entité_B']:
    sub = df_panel.xs(entity, level='entity')
    for method in ['mean', 'sum']:
        result = converter.aggregate_to_lower_frequency(sub, 'QE', method=method)
        print(f"\n  {entity} — {method} :")
        for date, row in result.iterrows():
            print(f"    {date.strftime('%Y-%m-%d')} → ventes={row['ventes']:.2f}, cout={row['cout']:.2f}")

In [ ]:
# Visualisation panel
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, entity in zip(axes, ['entité_A', 'entité_B']):
    sub = df_panel.xs(entity, level='entity')
    result_mean = converter.aggregate_to_lower_frequency(sub, 'QE', method='mean')
    result_sum = converter.aggregate_to_lower_frequency(sub, 'QE', method='sum')
    
    ax.plot(sub.index, sub['ventes'], 'o-', color=COLORS['original'],
            label='Mensuel (ventes)', markersize=4)
    ax.bar(result_mean.index, result_mean['ventes'], width=60, alpha=0.4,
           color=COLORS['aggregated'], label='Trimestriel (mean)')
    
    ax.set_title(f'{entity}', fontweight='bold')
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=45)

fig.suptitle('Agrégation panel mensuel → trimestriel', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. `interpolate_to_higher_frequency` — Séries temporelles

### 3.1 Cas nominal : interpolation trimestrielle → mensuelle

On crée une série trimestrielle et on interpole vers la fréquence mensuelle avec différentes méthodes.

In [ ]:
# Série trimestrielle sur 2 ans
dates_q = pd.date_range('2024-01-01', periods=8, freq='QS')
values_q = [100, 120, 115, 130, 125, 140, 135, 150]
series_q = pd.Series(values_q, index=dates_q, name='pib')

print("Série trimestrielle originale :")
print(series_q)

print_section("Interpolation trimestrielle → mensuelle")

for method in ['linear', 'nearest', 'cubic']:
    result = converter.interpolate_to_higher_frequency(series_q, 'MS', method=method)
    print(f"\n  Méthode '{method}' ({len(result)} points) :")
    print(f"    Premiers : {result.head(6).values.round(2).tolist()}")
    print(f"    Derniers : {result.tail(3).values.round(2).tolist()}")

In [ ]:
# Visualisation des différentes méthodes d'interpolation
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, method in zip(axes, ['linear', 'nearest', 'cubic']):
    result = converter.interpolate_to_higher_frequency(series_q, 'MS', method=method)
    
    ax.plot(result.index, result.values, '-', color=COLORS['interpolated'],
            alpha=0.7, label=f'Mensuel ({method})')
    ax.plot(series_q.index, series_q.values, 'o', color=COLORS['original'],
            markersize=8, label='Trimestriel (original)', zorder=5)
    ax.set_title(f"Interpolation '{method}'", fontweight='bold')
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=45)

fig.suptitle('Interpolation trimestrielle → mensuelle', fontweight='bold')
plt.tight_layout()
plt.show()

### 3.2 Cas critique : NaN en début de série et délais de publication en fin de série

Scénario réaliste :
- Les **premières périodes** ont des NaN (données pas encore disponibles historiquement)
- Les **dernières périodes** ont des NaN (délais de publication : les données les plus récentes ne sont pas encore publiées)

On examine l'effet de `fill_method` (`ffill`, `bfill`, `None`) sur ces NaN.

In [ ]:
# Série trimestrielle avec NaN en début et fin (délais de publication)
dates_q_pub = pd.date_range('2023-01-01', periods=10, freq='QS')
values_pub = [np.nan, np.nan, 105, 120, 115, 130, 125, 140, np.nan, np.nan]
series_pub = pd.Series(values_pub, index=dates_q_pub, name='pib_delayed')

print("Série trimestrielle avec NaN (délais de publication) :")
print(series_pub)
print(f"\nNaN en début : {series_pub.isna().values[:2].tolist()}")
print(f"NaN en fin   : {series_pub.isna().values[-2:].tolist()}")

In [ ]:
# Interpolation avec différents fill_method
print_section("Effet de fill_method sur les NaN")

results = {}
for fill_method in [None, 'ffill', 'bfill']:
    result = converter.interpolate_to_higher_frequency(
        series_pub, 'MS', method='linear', fill_method=fill_method
    )
    results[str(fill_method)] = result
    
    label = fill_method if fill_method else 'None'
    nan_count = result.isna().sum()
    print(f"\n  fill_method='{label}' ({len(result)} points, {nan_count} NaN) :")
    print(f"    Début (premiers 6 mois) :")
    for date, val in result.head(6).items():
        status = '  ← NaN' if pd.isna(val) else ''
        val_str = f'{val:.2f}' if not pd.isna(val) else 'NaN'
        print(f"      {date.strftime('%Y-%m-%d')} → {val_str}{status}")
    print(f"    Fin (derniers 6 mois) :")
    for date, val in result.tail(6).items():
        status = '  ← NaN' if pd.isna(val) else ''
        val_str = f'{val:.2f}' if not pd.isna(val) else 'NaN'
        print(f"      {date.strftime('%Y-%m-%d')} → {val_str}{status}")

In [ ]:
# Visualisation comparative des fill_method
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

fill_colors = {'None': COLORS['interpolated'], 'ffill': COLORS['ffill'], 'bfill': COLORS['bfill']}

for ax, (fill_label, result) in zip(axes, results.items()):
    color = fill_colors[fill_label]
    
    # Points interpolés (non-NaN)
    mask_valid = result.notna()
    ax.plot(result.index[mask_valid], result.values[mask_valid], '-',
            color=color, alpha=0.7, label=f'Mensuel interpolé')
    
    # Points NaN
    mask_nan = result.isna()
    if mask_nan.any():
        ax.scatter(result.index[mask_nan], [0] * mask_nan.sum(),
                   color=COLORS['nan'], marker='x', s=50, label='NaN restants', zorder=5)
    
    # Points originaux
    mask_orig_valid = series_pub.notna()
    ax.plot(series_pub.index[mask_orig_valid], series_pub.values[mask_orig_valid],
            'o', color=COLORS['original'], markersize=8, label='Original (valeurs)', zorder=6)
    
    mask_orig_nan = series_pub.isna()
    ax.scatter(series_pub.index[mask_orig_nan],
              [series_pub.dropna().mean()] * mask_orig_nan.sum(),
              color=COLORS['nan'], marker='s', s=60, label='Original (NaN)', zorder=6)
    
    # Zone de NaN en début et fin
    ax.axvspan(series_pub.index[0], series_pub.index[1], alpha=0.08, color='red',
               label='Zone NaN début')
    ax.axvspan(series_pub.index[-2], series_pub.index[-1], alpha=0.08, color='orange',
               label='Zone NaN fin (délai pub.)')
    
    ax.set_title(f"fill_method='{fill_label}'", fontweight='bold')
    ax.legend(fontsize=7, loc='upper left')
    ax.tick_params(axis='x', rotation=45)

fig.suptitle('Effet de fill_method sur l\'interpolation avec NaN (délais de publication)',
             fontweight='bold')
plt.tight_layout()
plt.show()

### 3.3 Analyse de l'effet de `ffill` et `bfill`

| `fill_method` | NaN en début de série | NaN en fin de série (délai de publication) |
|---|---|---|
| `None` | Restent NaN → interpolation linéaire ne peut pas les combler | Restent NaN → aucune extrapolation |
| `ffill` | Restent NaN (pas de valeur précédente) | Propagation de la dernière valeur connue → **risque de plateau artificiel** |
| `bfill` | Propagation de la première valeur connue vers le passé | Restent NaN (pas de valeur suivante) |

**Conclusion** : ni `ffill` ni `bfill` seul ne résout les deux côtés. Pour un cas réaliste avec des délais de publication, `ffill` est souvent préféré car il est plus important de prolonger la série vers le présent que de combler le passé.

In [ ]:
# Tableau comparatif détaillé : NaN restants par zone
print_section("NaN restants par zone et fill_method")

for fill_label, result in results.items():
    # Identification des zones
    debut = result.loc[:'2023-06-01']
    milieu = result.loc['2023-07-01':'2024-12-01']
    fin = result.loc['2025-01-01':]
    
    print(f"\n  fill_method='{fill_label}' :")
    print(f"    Zone début  (avant données) : {debut.isna().sum():>2} NaN / {len(debut)} points")
    print(f"    Zone milieu (données dispo)  : {milieu.isna().sum():>2} NaN / {len(milieu)} points")
    print(f"    Zone fin    (délai pub.)     : {fin.isna().sum():>2} NaN / {len(fin)} points")

---
## 4. `interpolate_to_higher_frequency` — Données de panel

On crée un panel trimestriel avec 2 entités ayant des profils de NaN différents :
- **Entité A** : NaN en fin de série (délai de publication classique)
- **Entité B** : NaN en début de série (série plus courte historiquement)

In [ ]:
# Panel trimestriel avec NaN asymétriques
dates_q_panel = pd.date_range('2023-01-01', periods=8, freq='QS')

# Entité A : NaN en fin (délai de publication)
vals_a = [100, 110, 105, 120, 115, 130, np.nan, np.nan]
df_qa = pd.DataFrame({'pib': vals_a}, index=dates_q_panel)
df_qa['entity'] = 'entité_A'

# Entité B : NaN en début (série courte)
vals_b = [np.nan, np.nan, 200, 210, 205, 220, 215, 230]
df_qb = pd.DataFrame({'pib': vals_b}, index=dates_q_panel)
df_qb['entity'] = 'entité_B'

# Assemblage
df_panel_q = pd.concat([df_qa, df_qb])
df_panel_q = df_panel_q.reset_index().rename(columns={'index': 'date'})
df_panel_q = df_panel_q.set_index(['entity', 'date'])

print("Panel trimestriel avec NaN asymétriques :")
print("\nEntité A (NaN en fin — délai pub.) :")
print(df_panel_q.xs('entité_A', level='entity'))
print("\nEntité B (NaN en début — série courte) :")
print(df_panel_q.xs('entité_B', level='entity'))

In [ ]:
# Interpolation par entité avec différents fill_method
print_section("Interpolation panel trimestriel → mensuel")

panel_results = {}
for entity in ['entité_A', 'entité_B']:
    sub = df_panel_q.xs(entity, level='entity')
    panel_results[entity] = {}
    
    for fill_method in [None, 'ffill', 'bfill']:
        result = converter.interpolate_to_higher_frequency(
            sub, 'MS', method='linear', fill_method=fill_method
        )
        panel_results[entity][str(fill_method)] = result
        
        label = fill_method if fill_method else 'None'
        nan_count = result.isna().sum().sum()
        print(f"\n  {entity} — fill_method='{label}' : {nan_count} NaN restants")

In [ ]:
# Visualisation panel : comparaison ffill vs bfill par entité
fig, axes = plt.subplots(2, 3, figsize=(18, 9))

for row, entity in enumerate(['entité_A', 'entité_B']):
    sub_orig = df_panel_q.xs(entity, level='entity')
    
    for col, fill_label in enumerate(['None', 'ffill', 'bfill']):
        ax = axes[row, col]
        result = panel_results[entity][fill_label]
        color = fill_colors[fill_label]
        
        # Interpolé
        mask_valid = result['pib'].notna()
        ax.plot(result.index[mask_valid], result['pib'].values[mask_valid], '-',
                color=color, alpha=0.7, label='Mensuel interpolé')
        
        # NaN restants
        mask_nan = result['pib'].isna()
        if mask_nan.any():
            y_nan = sub_orig['pib'].dropna().mean()
            ax.scatter(result.index[mask_nan], [y_nan] * mask_nan.sum(),
                       color=COLORS['nan'], marker='x', s=50, label='NaN', zorder=5)
        
        # Original
        orig_valid = sub_orig['pib'].notna()
        ax.plot(sub_orig.index[orig_valid], sub_orig['pib'].values[orig_valid],
                'o', color=COLORS['original'], markersize=8, label='Original', zorder=6)
        
        ax.set_title(f"{entity} — fill='{fill_label}'", fontweight='bold', fontsize=10)
        ax.legend(fontsize=7)
        ax.tick_params(axis='x', rotation=45)

fig.suptitle('Panel : interpolation Q→M avec NaN asymétriques par entité', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Synthèse et recommandations

### `aggregate_to_lower_frequency`

| Aspect | Comportement |
|---|---|
| Trimestres complets | Agrégation standard sans surprise |
| Trimestre incomplet | `resample` agrège sur les mois disponibles : `mean` correcte, `sum` sous-estimée |
| Détection | Utiliser `count` en parallèle pour identifier les périodes incomplètes |

### `interpolate_to_higher_frequency`

| `fill_method` | NaN début | NaN fin (délai pub.) | Cas d'usage |
|---|---|---|---|
| `None` | Restent NaN | Restent NaN | Préserver l'information de disponibilité |
| `ffill` | Restent NaN | Propagation dernière valeur | Prolonger la série vers le présent |
| `bfill` | Propagation première valeur | Restent NaN | Combler le passé (rarement souhaité) |